## EuroSAT Project
### Members: Katie Knox, Helen Xu, Ezra Rwakazooba
#### Submission date:  April 5th, 2026

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change on your computer for connection
import os
%cd "/content/drive/MyDrive/Colab Notebooks/EuroStats"
os.getcwd()

In [ ]:
import zipfile
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from pathlib import Path
from io import BytesIO

### 1. Data Loading, Processing, and Exploration

#### 1.1 Data Preparation

The [EuroSAT dataset](https://github.com/phelber/eurosat) is a benchmark land-use and land-cover (LULC) classification dataset derived from Sentinel-2 satellite imagery. It contains 27,000 geo-referenced, 64×64-pixel multispectral images across 10 land-cover classes. Each image has 13 spectral bands corresponding to the Sentinel-2 multispectral instrument (MSI).

The following steps are performed:
1. The EuroSAT multispectral `.zip` archive is extracted
2. All images and their class labels are indexed into a `(path, label)` sample list
3. Class distribution is assessed and visualized
4. One representative image per class is plotted as an RGB composite (using Sentinel-2 bands B04/B03/B02)

In [ ]:
ZIP_PATH = r"data/EuroSAT_MS.zip"
EXTRACT_DIR = Path("/content/EuroSAT_MS")

# Extract zip if not already done
if not EXTRACT_DIR.exists():
    print("Extracting zip...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR.parent)
    print("Done.")
else:
    print("Already extracted.")

In [ ]:
# Build list of (file_path, label) pairs from folder names
samples = []
classes = sorted([d.name for d in EXTRACT_DIR.iterdir() if d.is_dir()])
class_to_idx = {cls: i for i, cls in enumerate(classes)}

for cls in classes:
    cls_dir = EXTRACT_DIR / cls
    for img_path in sorted(cls_dir.glob("*.tif")):
        samples.append((img_path, class_to_idx[cls]))

print(f"Classes ({len(classes)}): {classes}")
print(f"Total images: {len(samples)}")

In [ ]:
from collections import Counter

label_counts = Counter(label for _, label in samples)
counts = [label_counts[class_to_idx[cls]] for cls in classes]
total = sum(counts)

for cls, idx in class_to_idx.items():
    class_pcts = np.round(label_counts[idx] / total * 100, 1)
    print(f"  {cls:25s}: {label_counts[idx]:,} images ({class_pcts}%)")

The dataset has *27000* samples with good class balance.
AnnualCrop, Forest, HerbaceousVegetation, Residential and SeaLake all balance (3,000 samples, ~11.1%), while Highway, Industrial,  PermanentCrop and River   (2500 samples ~9.3%) and Pasture being the least class (2000 samples ~7.4%)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(classes, counts, color="steelblue", edgecolor="white")

for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f"{count:,}\n({count / total:.1%})",
        ha="center",
        va="bottom",
        fontsize=8.5,
    )

mean_count = total // len(classes)
ax.axhline(
    mean_count,
    color="tomato",
    linestyle="--",
    linewidth=1,
    label=f"Mean ({mean_count:,})",
)
ax.set_title("EuroSAT MS – Class Distribution", fontsize=13, pad=12)
ax.set_ylabel("Number of images")
ax.set_ylim(0, max(counts) * 1.2)
ax.set_xticklabels(classes, rotation=30, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

#### Sample Images per Class

One RGB composite image (Sentinel-2 bands B04=Red, B03=Green, B02=Blue) is displayed for each of the 10 land-cover classes in a 2×5 grid.

In [ ]:
def load_image(path):
    """Load a multispectral GeoTIFF as a (bands, H, W) numpy array."""
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)  # shape: (13, 64, 64)

# Inspect a single image
img_path, label = samples[0]
img = load_image(img_path)
print(f"Image shape : {img.shape}  (bands, H, W)")
print(f"Label       : {label} ({classes[label]})")
print(f"Value range : {img.min():.0f} – {img.max():.0f}")

In [ ]:
# Sentinel-2 band reference
BAND_NAMES = [
    "B01 (Coastal)", "B02 (Blue)", "B03 (Green)", "B04 (Red)",
    "B05 (Red Edge 1)", "B06 (Red Edge 2)", "B07 (Red Edge 3)",
    "B08 (NIR)", "B08A (Narrow NIR)", "B09 (Water Vapor)",
    "B10 (SWIR Cirrus)", "B11 (SWIR 1)", "B12 (SWIR 2)"
]

# Display one sample per class using RGB (B04, B03, B02)
RED, GREEN, BLUE = 3, 2, 1  # band indices (0-based)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i, cls in enumerate(classes):
    # Grab first sample of this class
    path, _ = next((s, l) for s, l in samples if classes[l] == cls)
    img = load_image(path)

    # Build RGB composite and normalise to [0, 1]
    rgb = np.stack([img[RED], img[GREEN], img[BLUE]], axis=-1)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)

    axes[i].imshow(rgb)
    axes[i].set_title(cls, fontsize=10)
    axes[i].axis("off")

plt.suptitle("EuroSAT MS – one sample per class (RGB composite)", fontsize=13)
plt.tight_layout()
plt.show()

### 1.2 Data Augmentation

To avoid **data leakage**, the original images are **split into training and testing sets first**, and **augmentation is applied only to the training set**. This ensures that transformed versions of the same original image never appear in both splits.

The pipeline is:
1. Stratified 60/40 train/test split on the original 27,000 images
2. Augmentation is applied only to the 16,200 training images
3. The test set (10,800 images) remains unmodified

The augmentation methods used are:
- **Rotation** (random angle 0–180°): exposes the model to satellite imagery viewed from different orientations, which is critical since satellite images have no canonical "up"
- Zoom and shift functions were implemented but ultimately not applied to the final dataset, as the 64×64 spatial resolution left too much blank padding after those transforms

After augmentation, the total modeling dataset contains **43,200 samples** (32,400 train + 10,800 test). Class proportions are preserved identically in both splits thanks to stratified sampling.

 The data augmentation method we propose are the following;
<br>

<ol>
    <li> Rotating (0-180) degrees
    <li> Zoom
    <li> width shift
    <li> Height shift
</ol>


In [ ]:
from scipy.ndimage import rotate, shift
from scipy.ndimage import zoom as sp_zoom
from tqdm.notebook import tqdm

In [ ]:
def load_image(path: Path) -> np.ndarray:
    """Load a .tif or augmented .npy file as a (bands, H, W) float32 array."""
    path = Path(path)
    if path.suffix == ".npy":
        return np.load(path).astype(np.float32)
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)

In [ ]:
def augment_rotate(img: np.ndarray, angle: float = 30.0) -> np.ndarray:
    """Rotate image by angle degrees in the spatial (H, W) plane."""
    return rotate(img, angle, axes=(1, 2), reshape=False)


def augment_zoom(img: np.ndarray, factor: float = 1.2) -> np.ndarray:
    """Zoom in by factor and centre-crop back to the original spatial size."""
    _, h, w = img.shape
    zoomed = sp_zoom(img, [1.0, factor, factor])
    _, zh, zw = zoomed.shape
    ch, cw = (zh - h) // 2, (zw - w) // 2
    return zoomed[:, ch : ch + h, cw : cw + w]


def augment_width_shift(img: np.ndarray, shift_px: int = 8) -> np.ndarray:
    """Shift image horizontally by shift_px pixels."""
    return shift(img, [0, 0, shift_px])


def augment_height_shift(img: np.ndarray, shift_px: int = 8) -> np.ndarray:
    """Shift image vertically by shift_px pixels."""
    return shift(img, [0, shift_px, 0])

# for Size I will only use rotate and zoom, as the shifts cause too much blank space around the edges which is not ideal for a small 64x64 image. I will keep the shift functions in case I want to experiment with them later, but I will comment them out of the AUGMENTATIONS dict for now.
AUGMENTATIONS = {
    "rotate": augment_rotate,
    # "zoom": augment_zoom,
    #"width_shift": augment_width_shift,
    #"height_shift": augment_height_shift,
}

In [ ]:
# Split the ORIGINAL dataset first (before augmentation) to avoid data leakage
from sklearn.model_selection import train_test_split

orig_paths = [path for path, _ in samples]
orig_labels = [label for _, label in samples]

train_paths, test_paths, y_train_orig, y_test_orig = train_test_split(
    orig_paths,
    orig_labels,
    test_size=0.4,
    random_state=42,
    stratify=orig_labels
)

train_samples_orig = list(zip(train_paths, y_train_orig))
test_samples_orig = list(zip(test_paths, y_test_orig))

print(f"Original training samples: {len(train_samples_orig):,}")
print(f"Original testing samples : {len(test_samples_orig):,}")

In [ ]:
# NO NEED TO RUN --> Data created!
# Create a separate augmentation folder for TRAINING data only
AUG_DIR = Path("data/augmented_train_only")
AUG_DIR.mkdir(parents=True, exist_ok=True)

for cls in classes:
    (AUG_DIR / cls).mkdir(parents=True, exist_ok=True)

train_aug_samples = []

for path, lbl in tqdm(train_samples_orig, desc="Augmenting training set only"):
    cls = classes[lbl]
    img = load_image(path)
    stem = Path(path).stem

    for aug_name, aug_fn in AUGMENTATIONS.items():
        out_path = AUG_DIR / cls / f"{stem}_{aug_name}.npy"
        if not out_path.exists():
            np.save(out_path, aug_fn(img))
        train_aug_samples.append((str(out_path), lbl))

print(f"Augmented training samples created/loaded: {len(train_aug_samples):,}")

In [ ]:
# RUN THIS to GET the DATA
from pathlib import Path

AUG_DIR = Path("data/augmented_train_only")

train_aug_samples = []

for lbl, cls in enumerate(classes):
    cls_dir = AUG_DIR / cls
    aug_files = sorted(cls_dir.glob("*.npy"))
    train_aug_samples.extend([(str(p), lbl) for p in aug_files])

print("Reloaded augmented training samples:", len(train_aug_samples))
print("Example:", train_aug_samples[0])

In [ ]:
# Final modeling splits
train_samples = train_samples_orig + train_aug_samples
test_samples = test_samples_orig

print(f"Train (original only) : {len(train_samples_orig):,}")
print(f"Train (augmented only): {len(train_aug_samples):,}")
print(f"Train (final total)   : {len(train_samples):,}")
print(f"Test  (unchanged)     : {len(test_samples):,}")

In [ ]:
# Combined dataset summary for reference
modeling_samples = train_samples + test_samples

print(f"Original dataset size : {len(samples):,}")
print(f"Modeling dataset size : {len(modeling_samples):,}")

In [ ]:
# Class distribution after augmenting ONLY the training split
train_label_counts = Counter(label for _, label in train_samples)
test_label_counts = Counter(label for _, label in test_samples)
all_label_counts = Counter(label for _, label in modeling_samples)

print("Training distribution:")
for cls, idx in class_to_idx.items():
    pct = np.round(train_label_counts[idx] / len(train_samples) * 100, 1)
    print(f"  {cls:25s}: {train_label_counts[idx]:,} images ({pct}%)")

print("\nTesting distribution:")
for cls, idx in class_to_idx.items():
    pct = np.round(test_label_counts[idx] / len(test_samples) * 100, 1)
    print(f"  {cls:25s}: {test_label_counts[idx]:,} images ({pct}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

label_series = [label for _, label in modeling_samples]
ax.hist(
    label_series,
    bins=len(classes),
    range=(-0.5, len(classes) - 0.5),
    color="steelblue",
    edgecolor="white",
)

all_total = len(modeling_samples)
all_mean = all_total // len(classes)

ax.axvline(
    np.mean(label_series),
    color="tomato",
    linestyle="--",
    linewidth=1,
    label=f"Mean ({all_mean:,})",
)
ax.set_title("EuroSAT MS – Class Distribution After Train-Only Augmentation", fontsize=13, pad=12)
ax.set_ylabel("Number of images")
ax.set_xticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=30, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import random

def load_any_image(path):
    """
    Load either:
    - original .tif image -> return RGB composite from MS bands
    - augmented .npy image -> load directly
    Returns image in (H, W, 3), normalized to [0, 1]
    """
    path = Path(path)

    if path.suffix == ".npy":
        img = np.load(path)

        if img.ndim == 3 and img.shape[0] == 13:
            rgb = np.stack([img[3], img[2], img[1]], axis=-1)
        elif img.ndim == 3 and img.shape[-1] == 3:
            rgb = img
        else:
            raise ValueError(f"Unexpected .npy shape: {img.shape}")
    else:
        with rasterio.open(path) as src:
            img = src.read().astype(np.float32)
        rgb = np.stack([img[3], img[2], img[1]], axis=-1)

    rgb = rgb.astype(np.float32)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
    return rgb

# show three random images from the TRAINING set after augmentation
random_samples = random.sample(train_samples, 3)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (img_path, label) in zip(axes, random_samples):
    rgb = load_any_image(img_path)
    ax.imshow(rgb)
    ax.set_title(classes[label], fontsize=10)
    ax.axis("off")

plt.suptitle("Three Random Images from the Augmented Training Set", fontsize=13)
plt.tight_layout()
plt.show()

#### Preparing Grayscale Data for Traditional ML

For the traditional machine learning models (SVMs and Random Forest), a **grayscale** representation is used. Each 13-band multispectral image is converted to a single-channel grayscale image using luminance weighting on the Sentinel-2 RGB bands (B04=Red, B03=Green, B02=Blue):

`gray = 0.2989 × R + 0.5870 × G + 0.1140 × B`

Each 64×64 grayscale image is then **flattened** into a 1D feature vector of 4,096 values (64 × 64), producing an `(n, 4096)` feature matrix suitable for scikit-learn models.

For the traditional ML section (Section 2), only the three target classes — **Forest**, **Residential**, and **Industrial** — are loaded, yielding 10,200 training and 3,400 test samples.

In [ ]:
import dask
from dask import delayed
from dask.diagnostics import ProgressBar
from sklearn.model_selection import train_test_split

In [ ]:
# NO NEED TO RUN
# Too Many Samples, ignore the following three chunk, only use the need samples

'''
@delayed
def load_flat(path: Path) -> np.ndarray:
    """Load one image and flatten to a 1-D (bands × H × W) float32 array."""
    return load_image(path).flatten()


y = np.array([lbl for _, lbl in all_samples], dtype=np.int32)
lazy_rows = [load_flat(path) for path, _ in all_samples]

with ProgressBar():
    rows = dask.compute(*lazy_rows, scheduler="threads")

X = np.stack(rows).astype(np.float32)

print(f"X shape : {X.shape}  (n={X.shape[0]:,} samples, p={X.shape[1]:,} pixels)")
print(f"y shape : {y.shape}")
print(f"RAM     : {X.nbytes / 1e9:.1f} GB")

'''

In [ ]:
# NO NEED TO RUN
'''
indices = np.arange(len(all_samples))

idx_train, idx_test, y_train, y_test = train_test_split(
    indices, y, test_size=0.4, stratify=y, random_state=42
)

X_train, X_test = X[idx_train], X[idx_test]

print(f"Train : X={X_train.shape},  y={y_train.shape}")
print(f"Test  : X={X_test.shape},   y={y_test.shape}")
'''

In [ ]:
# NO NEED TO RUN
'''
# Reshape to (n, 13, 64, 64) to isolate visible RGB bands
n = X.shape[0]
X_cube = X.reshape(n, 13, 64, 64)

# Luminance-weighted grayscale from B04 (Red), B03 (Green), B02 (Blue)
R = X_cube[:, 3].reshape(n, -1)
G = X_cube[:, 2].reshape(n, -1)
B = X_cube[:, 1].reshape(n, -1)

X_gray = (0.2989 * R + 0.5870 * G + 0.1140 * B).astype(np.float32)

X_gray_train = X_gray[idx_train]
X_gray_test = X_gray[idx_test]

print(f"X_gray       : {X_gray.shape}  ({X_gray.nbytes / 1e9:.1f} GB)")
print(f"X_gray_train : {X_gray_train.shape}")
print(f"X_gray_test  : {X_gray_test.shape}")
'''

In [ ]:
# Extract Forest / Residential / Industrial separately for train and test
target_classes = ["Forest", "Residential", "Industrial"]
target_idx = [class_to_idx[c] for c in target_classes]

train_subset_samples = [(path, lbl) for path, lbl in train_samples if lbl in target_idx]
test_subset_samples = [(path, lbl) for path, lbl in test_samples if lbl in target_idx]

print("Selected classes:", target_classes)
print("Class indices:", target_idx)
print("Train subset size:", len(train_subset_samples))
print("Test subset size :", len(test_subset_samples))

In [ ]:
# check path
print(samples[0][0])
print(train_aug_samples[0][0])

print(Path(samples[0][0]).exists())
print(Path(train_aug_samples[0][0]).exists())

In [ ]:
def load_ms_array(path):
    path = Path(path)

    if path.suffix == ".npy":
        img = np.load(path).astype(np.float32)
        if img.ndim == 3 and img.shape[0] == 13:
            return img
        else:
            raise ValueError(f"Unexpected npy shape for MS data: {img.shape}")

    else:
        with rasterio.open(path) as src:
            img = src.read().astype(np.float32)   # (13, H, W)
        return img

In [ ]:
from tqdm import tqdm

def ms_to_rgb(ms_img):
    # Sentinel-2: B04, B03, B02 -> R, G, B
    return np.stack([ms_img[3], ms_img[2], ms_img[1]], axis=-1)

def rgb_to_gray(rgb_img):
    return (
        0.2989 * rgb_img[..., 0] +
        0.5870 * rgb_img[..., 1] +
        0.1140 * rgb_img[..., 2]
    ).astype(np.float32)

def build_gray_dataset(sample_list, desc="Converting MS to grayscale"):
    X_gray, y_out = [], []
    for path, lbl in tqdm(sample_list, desc=desc):
        ms = load_ms_array(path)
        rgb = ms_to_rgb(ms)
        gray = rgb_to_gray(rgb)
        X_gray.append(gray)
        y_out.append(lbl)
    return np.stack(X_gray), np.array(y_out, dtype=np.int32)

X_gray_train, y_train = build_gray_dataset(train_subset_samples, desc="Preparing TRAIN grayscale data")
X_gray_test, y_test = build_gray_dataset(test_subset_samples, desc="Preparing TEST grayscale data")

print("X_gray_train shape:", X_gray_train.shape)
print("X_gray_test shape :", X_gray_test.shape)
print("y_train shape     :", y_train.shape)
print("y_test shape      :", y_test.shape)

In [ ]:
# flatten
X_train = X_gray_train.reshape(X_gray_train.shape[0], -1)
X_test = X_gray_test.reshape(X_gray_test.shape[0], -1)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

In [ ]:
# The split has already been done BEFORE augmentation, so no second split is needed.
print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

In [ ]:
# Grayscale Faltterned
# store the data in case need to re-run
np.savez(
    "data/dataset_grayscale_flatterned_ML.npz",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

In [ ]:
# call the data
data = np.load("data/dataset_grayscale_flatterned_ML.npz")

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

### 2. Traditional Machine Learning

For this section we focus on three EuroSAT classes: **Forest (F)**, **Residential (R)**, and **Industrial (I)**. The grayscale, flattened dataset prepared in Section 1 is used throughout. The three class indices in the dataset are: Forest=1, Residential=7, Industrial=4.

#### 2.1 Binary Support Vector Machine

Three binary SVM classifiers with a **linear kernel** and default parameters are trained on the grayscale flattened images, one for each pairwise class combination: Forest vs Residential, Forest vs Industrial, and Residential vs Industrial.

For each classifier, we report: accuracy on the test set, ROC curve, AUC, and one misclassified example (with true and predicted label).

In [ ]:
# load packages
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_curve, auc

In [ ]:
# check target class id
target_classes = ["Forest", "Residential", "Industrial"]

for cls in target_classes:
    print(cls, "->", class_to_idx[cls])

In [ ]:
# define model
def run_binary_svm(X_train, X_test, y_train, y_test, class_a, class_b, classes):
    train_mask = np.isin(y_train, [class_a, class_b])
    test_mask = np.isin(y_test, [class_a, class_b])

    X_train_bin = X_train[train_mask]
    X_test_bin = X_test[test_mask]
    y_train_bin = y_train[train_mask]
    y_test_bin = y_test[test_mask]

    y_train_bin01 = np.where(y_train_bin == class_a, 0, 1)
    y_test_bin01 = np.where(y_test_bin == class_a, 0, 1)

    svm = SVC(kernel="linear", random_state=42)
    svm.fit(X_train_bin, y_train_bin01)

    y_pred = svm.predict(X_test_bin)
    y_score = svm.decision_function(X_test_bin)

    acc = accuracy_score(y_test_bin01, y_pred)
    fpr, tpr, _ = roc_curve(y_test_bin01, y_score)
    roc_auc = auc(fpr, tpr)

    mis_idx = np.where(y_pred != y_test_bin01)[0]
    mis_example = None

    if len(mis_idx) > 0:
        i = mis_idx[0]
        mis_example = {
            "image_flat": X_test_bin[i],
            "true_label_name": classes[class_a] if y_test_bin01[i] == 0 else classes[class_b],
            "pred_label_name": classes[class_a] if y_pred[i] == 0 else classes[class_b],
        }

    return {
        "model": svm,
        "class_a_name": classes[class_a],
        "class_b_name": classes[class_b],
        "accuracy": acc,
        "fpr": fpr,
        "tpr": tpr,
        "auc": roc_auc,
        "mis_example": mis_example,
    }

In [ ]:
# call the model
FOREST = 1
RESIDENTIAL = 7
INDUSTRIAL = 4

from tqdm.notebook import tqdm

binary_tasks = [
    ("Forest vs Residential", FOREST, RESIDENTIAL),
    ("Forest vs Industrial", FOREST, INDUSTRIAL),
    ("Residential vs Industrial", RESIDENTIAL, INDUSTRIAL),
]

binary_results = {}

for name, class_a, class_b in tqdm(binary_tasks, desc="Training binary SVMs"):
    binary_results[name] = run_binary_svm(
        X_train, X_test, y_train, y_test,
        class_a, class_b, classes
    )

svm_fr = binary_results["Forest vs Residential"]
svm_fi = binary_results["Forest vs Industrial"]
svm_ri = binary_results["Residential vs Industrial"]

In [ ]:
# Model result: accuracy and AUC
for result in [svm_fr, svm_fi, svm_ri]:
    print(f"{result['class_a_name']} vs {result['class_b_name']}")
    print(f"  Accuracy: {result['accuracy']:.4f}")
    print(f"  AUC     : {result['auc']:.4f}")
    print()


**Binary SVM Results:**

| Pair | Accuracy | AUC |
|---|---|---|
| Forest vs Residential | 0.9487 | 0.9920 |
| Forest vs Industrial | 0.9964 | 1.0000 |
| Residential vs Industrial | 0.6945 | 0.7294 |

**Interpretation:** Forest is the most spectrally distinct class — it is nearly perfectly separable from Industrial (AUC = 1.0) and very well-separated from Residential (AUC = 0.992). The Residential vs Industrial pair is the hardest (AUC = 0.729), which makes intuitive sense: both classes contain built-up urban structures and share similar spectral signatures in greyscale. The linear SVM with a flat pixel feature vector struggles to distinguish the spatial texture patterns that separate these two classes.



In [ ]:
# ROC curves
plt.figure(figsize=(7, 6))

for result in [svm_fr, svm_fi, svm_ri]:
    plt.plot(
        result["fpr"],
        result["tpr"],
        label=f"{result['class_a_name']} vs {result['class_b_name']} (AUC = {result['auc']:.3f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Binary Linear SVM Classifiers")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# missed classification
def plot_misclassified_example(result):
    mis = result["mis_example"]

    if mis is None:
        print(f"No misclassified example found for {result['class_a_name']} vs {result['class_b_name']}.")
        return

    img = mis["image_flat"].reshape(64, 64)

    plt.figure(figsize=(4, 4))
    plt.imshow(img, cmap="gray")
    plt.title(
        f"{result['class_a_name']} vs {result['class_b_name']}\n"
        f"True: {mis['true_label_name']} | Pred: {mis['pred_label_name']}"
    )
    plt.axis("off")
    plt.show()

for result in [svm_fr, svm_fi, svm_ri]:
    plot_misclassified_example(result)

In [ ]:
# save the result
import joblib
from pathlib import Path

SAVE_DIR = Path("data/saved_models")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(svm_fr["model"], SAVE_DIR / "svm_forest_vs_residential.joblib")
joblib.dump(svm_fi["model"], SAVE_DIR / "svm_forest_vs_industrial.joblib")
joblib.dump(svm_ri["model"], SAVE_DIR / "svm_residential_vs_industrial.joblib")

results_to_save = {
    "Forest_vs_Residential": {
        "accuracy": svm_fr["accuracy"],
        "auc": svm_fr["auc"],
        "fpr": svm_fr["fpr"],
        "tpr": svm_fr["tpr"],
        "mis_example": svm_fr["mis_example"],
    },
    "Forest_vs_Industrial": {
        "accuracy": svm_fi["accuracy"],
        "auc": svm_fi["auc"],
        "fpr": svm_fi["fpr"],
        "tpr": svm_fi["tpr"],
        "mis_example": svm_fi["mis_example"],
    },
    "Residential_vs_Industrial": {
        "accuracy": svm_ri["accuracy"],
        "auc": svm_ri["auc"],
        "fpr": svm_ri["fpr"],
        "tpr": svm_ri["tpr"],
        "mis_example": svm_ri["mis_example"],
    },
}

joblib.dump(results_to_save, SAVE_DIR / "binary_svm_results.joblib")

In [ ]:
# To extract the results
'''
split_info = {
    "FOREST": FOREST,
    "RESIDENTIAL": RESIDENTIAL,
    "INDUSTRIAL": INDUSTRIAL,
}

joblib.dump(split_info, SAVE_DIR / "label_info.joblib")
'''

#### 2.2 Multiclass, Majority-Vote SVM

The three binary SVMs from Section 2.1 are combined into a **three-class classifier** using a one-vs-one majority voting scheme. Each binary classifier casts a vote for one of its two classes, and the class with the most votes wins. Ties are broken by accumulated decision-function scores (converted to probabilities via sigmoid). ROC curves and AUCs are computed using a one-vs-rest strategy for each class.

In [ ]:
def get_binary_votes_and_scores(model_dict, X, class_a, class_b):
    """
    For one binary SVM, return:
    - predicted class labels (class_a or class_b)
    - score contribution for class_a and class_b

    Output:
    pred_labels: (N,)
    score_a: (N,)
    score_b: (N,)
    """
    model = model_dict["model"]

    # prediction in 0/1 space
    pred01 = model.predict(X)
    pred_labels = np.where(pred01 == 0, class_a, class_b)

    # score
    if hasattr(model, "predict_proba"):
        prob_b = model.predict_proba(X)[:, 1]
        score_b = prob_b
        score_a = 1 - prob_b
    else:
        # if using decision_function instead of predict_proba
        d = model.decision_function(X)
        prob_b = 1 / (1 + np.exp(-d))   # sigmoid to map into (0,1)
        score_b = prob_b
        score_a = 1 - prob_b

    return pred_labels, score_a, score_b

In [ ]:
def multiclass_majority_vote(X, svm_fr, svm_fi, svm_ri):
    """
    Combine 3 binary one-vs-one SVMs into a 3-class classifier
    using majority voting, with score-based tie breaking.

    Returns:
    y_pred_final: final predicted labels in original class ids
    vote_matrix: vote counts per class
    score_matrix: accumulated confidence scores per class
    """
    class_list = [FOREST, RESIDENTIAL, INDUSTRIAL]
    class_to_col = {FOREST: 0, RESIDENTIAL: 1, INDUSTRIAL: 2}

    n = X.shape[0]
    vote_matrix = np.zeros((n, 3), dtype=int)
    score_matrix = np.zeros((n, 3), dtype=float)

    # Forest vs Residential
    pred_fr, score_f_fr, score_r_fr = get_binary_votes_and_scores(svm_fr, X, FOREST, RESIDENTIAL)
    for i, pred in enumerate(pred_fr):
        vote_matrix[i, class_to_col[pred]] += 1
    score_matrix[:, class_to_col[FOREST]] += score_f_fr
    score_matrix[:, class_to_col[RESIDENTIAL]] += score_r_fr

    # Forest vs Industrial
    pred_fi, score_f_fi, score_i_fi = get_binary_votes_and_scores(svm_fi, X, FOREST, INDUSTRIAL)
    for i, pred in enumerate(pred_fi):
        vote_matrix[i, class_to_col[pred]] += 1
    score_matrix[:, class_to_col[FOREST]] += score_f_fi
    score_matrix[:, class_to_col[INDUSTRIAL]] += score_i_fi

    # Residential vs Industrial
    pred_ri, score_r_ri, score_i_ri = get_binary_votes_and_scores(svm_ri, X, RESIDENTIAL, INDUSTRIAL)
    for i, pred in enumerate(pred_ri):
        vote_matrix[i, class_to_col[pred]] += 1
    score_matrix[:, class_to_col[RESIDENTIAL]] += score_r_ri
    score_matrix[:, class_to_col[INDUSTRIAL]] += score_i_ri

    # majority vote with tie-breaking by score sum
    y_pred_final = []
    for i in range(n):
        max_vote = vote_matrix[i].max()
        candidates = np.where(vote_matrix[i] == max_vote)[0]

        if len(candidates) == 1:
            chosen_col = candidates[0]
        else:
            # tie break using highest accumulated score
            chosen_col = candidates[np.argmax(score_matrix[i, candidates])]

        y_pred_final.append(class_list[chosen_col])

    y_pred_final = np.array(y_pred_final)
    return y_pred_final, vote_matrix, score_matrix

In [ ]:
y_pred_mv, vote_matrix, score_matrix = multiclass_majority_vote(
    X_test, svm_fr, svm_fi, svm_ri
)

acc_mv = accuracy_score(y_test, y_pred_mv)
print(f"Majority-vote multiclass SVM accuracy: {acc_mv:.4f}")

In [ ]:
# binarize true labels in the same class order
from sklearn.preprocessing import label_binarize
y_test_bin = label_binarize(y_test, classes=[FOREST, RESIDENTIAL, INDUSTRIAL])

fpr = {}
tpr = {}
roc_auc = {}

class_names_3 = [classes[FOREST], classes[RESIDENTIAL], classes[INDUSTRIAL]]

for i, cls_name in enumerate(class_names_3):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], score_matrix[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(7, 6))
for i, cls_name in enumerate(class_names_3):
    plt.plot(
        fpr[i], tpr[i],
        label=f"{cls_name} vs Rest (AUC = {roc_auc[i]:.3f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Majority-Vote Multiclass SVM")
plt.legend()
plt.tight_layout()
plt.show()

for i, cls_name in enumerate(class_names_3):
    print(f"{cls_name} AUC: {roc_auc[i]:.4f}")


**Majority-Vote Multiclass SVM Results:**

- **Overall Accuracy:** 0.7765
- **Per-Class AUCs:** Forest = 0.9985, Residential = 0.8024, Industrial = 0.9060

**Interpretation:** The majority vote classifier achieves 77.65% accuracy across three classes — a substantial improvement over the ~69% binary Residential vs Industrial pair, because the Forest votes help anchor the overall decision boundary. Forest remains the most distinguishable class (AUC 0.999). Residential is the most confused class (AUC 0.802), often misclassified as Industrial. This is consistent with the binary SVM results showing that these two built-up classes are the hardest to separate in grayscale pixel space.



In [ ]:
def plot_one_misclassified_per_true_class(X_test, y_test, y_pred_mv, classes):
    target_classes = [FOREST, RESIDENTIAL, INDUSTRIAL]

    for cls in target_classes:
        mis_idx = np.where((y_test == cls) & (y_pred_mv != y_test))[0]

        if len(mis_idx) == 0:
            print(f"No misclassified example found for true class = {classes[cls]}.")
            continue

        i = mis_idx[0]
        img = X_test[i].reshape(64, 64)

        plt.figure(figsize=(4, 4))
        plt.imshow(img, cmap="gray")
        plt.title(
            f"True: {classes[y_test[i]]}\nPred: {classes[y_pred_mv[i]]}"
        )
        plt.axis("off")
        plt.show()

plot_one_misclassified_per_true_class(X_test, y_test, y_pred_mv, classes)

#### 2.3 Multiclass Random Forest

A **Random Forest** classifier (`n_estimators=200`) is trained on the three-class grayscale dataset. The model is applied to the test set, and we report accuracy, a confusion matrix, and one misclassified example per class. We also compare performance across 50, 100, and 200 trees.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {rf_acc:.4f}")

In [ ]:
label_order = [FOREST, RESIDENTIAL, INDUSTRIAL]
label_names = [classes[FOREST], classes[RESIDENTIAL], classes[INDUSTRIAL]]

cm = confusion_matrix(y_test, y_pred_rf, labels=label_order)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix: Random Forest")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
def plot_rf_misclassified_examples(X_test, y_test, y_pred, classes, class_list):
    for cls in class_list:
        mis_idx = np.where((y_test == cls) & (y_pred != y_test))[0]

        if len(mis_idx) == 0:
            print(f"No misclassified example found for true class = {classes[cls]}.")
            continue

        i = mis_idx[0]
        img = X_test[i].reshape(64, 64)

        plt.figure(figsize=(4, 4))
        plt.imshow(img, cmap="gray")
        plt.title(
            f"True: {classes[y_test[i]]}\nPred: {classes[y_pred[i]]}"
        )
        plt.axis("off")
        plt.show()

plot_rf_misclassified_examples(
    X_test, y_test, y_pred_rf,
    classes,
    [FOREST, RESIDENTIAL, INDUSTRIAL]
)

In [ ]:
for cls in [FOREST, RESIDENTIAL, INDUSTRIAL]:
    mis_idx = np.where((y_test == cls) & (y_pred_rf != y_test))[0]

    if len(mis_idx) == 0:
        print(f"{classes[cls]}: no misclassified example found.")
    else:
        i = mis_idx[0]
        print(
            f"{classes[cls]} misclassified example -> "
            f"True: {classes[y_test[i]]}, Predicted: {classes[y_pred_rf[i]]}"
        )

In [ ]:
# compare different estimator and different result
n_list = [50, 100, 200]

label_order = [FOREST, RESIDENTIAL, INDUSTRIAL]
label_names = [classes[FOREST], classes[RESIDENTIAL], classes[INDUSTRIAL]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, n in zip(axes, n_list):
    # train model
    rf = RandomForestClassifier(
        n_estimators=n,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    # predict
    y_pred = rf.predict(X_test)

    # accuracy
    acc = accuracy_score(y_test, y_pred)

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred, labels=label_order)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=label_names
    )

    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"n_estimators = {n}\nAcc = {acc:.3f}")
    ax.set_xticklabels(label_names, rotation=20)

plt.tight_layout()
plt.show()


**Random Forest Results:**

- **Accuracy (200 trees):** 0.9206

The Random Forest substantially outperforms both the binary SVMs and the majority-vote SVM, achieving over 92% accuracy. Random Forests capture non-linear decision boundaries and can leverage multi-scale texture information implicit in the flattened pixel vectors. Increasing the number of trees from 50 → 100 → 200 yields modest but consistent accuracy gains, as more trees reduce variance and improve ensemble stability. The confusion matrix shows that the primary remaining confusion is still between **Residential** and **Industrial**, consistent with the SVM results.

</font>

### 3. Deep Learning

For this section, **all 10 EuroSAT classes** are used (no subsetting to three classes). Three greyscale fully connected networks are trained first (Sections 3.1.1–3.1.3), followed by an ensemble (3.1.4), a CNN on RGB images (3.2.1), an advanced CNN (3.2.2), and finally the advanced CNN applied to RGB + NIR multispectral images (3.3).

#### 3.1 Greyscale Images

The same luminance-weighted greyscale images used in Section 2 are reloaded, but now for all 10 classes. Images are flattened to 4,096-dimensional vectors, shuffled, min-max normalized to [0, 1], and one-hot encoded before being fed into the neural networks.

**Pre-processing rationale:** Neural networks with softmax output require integer class labels to be one-hot encoded (categorical) because the cross-entropy loss operates on probability vectors. Pixel normalization to [0, 1] ensures stable gradient flow and prevents large raw pixel values from dominating weight updates early in training.

In [ ]:
# load grayscale data for the FULL 10-class problem using the leakage-safe split
from tqdm.notebook import tqdm
import numpy as np

def ms_to_rgb(ms_img):
    return np.stack([ms_img[3], ms_img[2], ms_img[1]], axis=-1)

def rgb_to_gray(rgb_img):
    return (
        0.2989 * rgb_img[..., 0] +
        0.5870 * rgb_img[..., 1] +
        0.1140 * rgb_img[..., 2]
    ).astype(np.float32)

def build_gray_dataset(sample_list, desc="Preparing grayscale dataset"):
    X_gray, y_out = [], []
    for path, lbl in tqdm(sample_list, desc=desc):
        ms = load_ms_array(path)
        rgb = ms_to_rgb(ms)
        gray = rgb_to_gray(rgb)
        X_gray.append(gray)
        y_out.append(lbl)
    return np.stack(X_gray), np.array(y_out, dtype=np.int32)

X_gray_train_full, y_train_full = build_gray_dataset(
    train_samples, desc="Preparing full TRAIN grayscale dataset"
)
X_gray_test_full, y_test_full = build_gray_dataset(
    test_samples, desc="Preparing full TEST grayscale dataset"
)

print("X_gray_train_full shape:", X_gray_train_full.shape)
print("X_gray_test_full shape :", X_gray_test_full.shape)
print("y_train_full shape     :", y_train_full.shape)
print("y_test_full shape      :", y_test_full.shape)

In [ ]:
# Flatten
X_train_gray_flat = X_gray_train_full.reshape(X_gray_train_full.shape[0], -1)
X_test_gray_flat = X_gray_test_full.reshape(X_gray_test_full.shape[0], -1)

print("X_train_gray_flat shape:", X_train_gray_flat.shape)
print("X_test_gray_flat shape :", X_test_gray_flat.shape)

In [ ]:
# No new split here — data were already split before augmentation
print(X_train_gray_flat.shape, X_test_gray_flat.shape)
print(y_train_full.shape, y_test_full.shape)

In [ ]:
X_train_gray_flat, y_train_full = shuffle(
    X_train_gray_flat,
    y_train_full,
    random_state=42
)

In [ ]:
# Normalized
train_max = X_train_gray_flat.max()

X_train_gray_flat_nn = X_train_gray_flat / (train_max + 1e-8)
X_test_gray_flat_nn = X_test_gray_flat / (train_max + 1e-8)

In [ ]:
# one-hot encoding
from tensorflow.keras.utils import to_categorical

num_classes = 10
y_train_cat = to_categorical(y_train_full, num_classes=num_classes)
y_test_cat = to_categorical(y_test_full, num_classes=num_classes)

print(y_train_cat.shape, y_test_cat.shape)

In [ ]:
check_class_balance(np.argmax(y_train_cat, axis=1))

##### 3.1.1 Single-Layer Neural Network

A minimal fully-connected network is implemented: **Input (4096) → Dense(10, softmax)**. This is equivalent to multinomial logistic regression — there is no hidden layer and no non-linearity before the output.

The model has **40,970 trainable parameters** (4096 × 10 weights + 10 biases). It is trained for 50 epochs using the Adam optimizer with categorical cross-entropy loss and a 20% validation split.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

In [ ]:
single_layer_model = keras.Sequential([
    layers.Input(shape=(4096,)),
    layers.Dense(len(classes), activation="softmax")
])

single_layer_model.summary()

In [ ]:
single_layer_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_single = single_layer_model.fit(
    X_train_gray_flat_nn,
    y_train_cat,
    validation_split=0.2,
    epochs=50,
    batch_size=128,
    verbose=1
)

In [ ]:
test_loss_single, test_acc_single = single_layer_model.evaluate(
    X_test_gray_flat_nn,
    y_test_cat,
    verbose=0
)

print(f"Single-Layer NN Test Accuracy: {test_acc_single:.4f}")
print(f"Single-Layer NN Test Loss: {test_loss_single:.4f}")


**Single-Layer NN Results:**

- **Test Accuracy:** 0.3055 (30.55%)

This is only modestly above random chance (10% for 10 classes), which is expected: a linear model cannot capture the complex, non-linear spectral-spatial patterns that distinguish 10 land-cover classes. The training accuracy plateaus around 32% and does not improve much with more epochs, suggesting the model is capacity-limited rather than simply underfit. One-hot encoding was necessary because the softmax output layer produces a 10-dimensional probability vector that must be compared to a one-hot target via categorical cross-entropy.



In [ ]:
import matplotlib.pyplot as plt
# Accuracy visualization
plt.figure(figsize=(6, 4))
plt.plot(history_single.history["accuracy"], label="Train Accuracy")
plt.plot(history_single.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Single-Layer Neural Network Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Loss visualization
plt.figure(figsize=(6, 4))
plt.plot(history_single.history["loss"], label="Train Loss")
plt.plot(history_single.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Single-Layer Neural Network Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

from tensorflow.keras.utils import plot_model

plot_model(
    single_layer_model,
    show_shapes=True,
    show_layer_names=True,
    dpi=70
)


##### 3.1.2 Two-Layer Neural Network

A hidden layer is added: **Input (4096) → Dense(128, ReLU) → Dense(10, softmax)**. This gives the network 525,706 trainable parameters — a 12.8× increase over the single-layer model. The hidden layer with ReLU activation allows the model to learn non-linear feature combinations before the final classification.

In [ ]:
# from tensorflow import keras
from tensorflow.keras import layers

two_layer_model = keras.Sequential([
    layers.Input(shape=(4096,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(len(classes), activation="softmax")
])

two_layer_model.summary()
from tensorflow import keras
from tensorflow.keras import layers

two_layer_model = keras.Sequential([
    layers.Input(shape=(4096,)),
    layers.Dense(128, activation="relu"), # hidden layer 128 nodes relu
    layers.Dense(len(classes), activation="softmax") # output layer 10 nodes
])

two_layer_model.summary()

In [ ]:
# complie model
two_layer_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_two = two_layer_model.fit(
    X_train_gray_flat_nn,
    y_train_cat,
    validation_split=0.2,
    epochs=50,
    batch_size=128,
    verbose=1
)

In [ ]:
test_loss_two, test_acc_two = two_layer_model.evaluate(
    X_test_gray_flat_nn,
    y_test_cat,
    verbose=0
)

print(f"Two-Layer NN Test Accuracy: {test_acc_two:.4f}")
print(f"Two-Layer NN Test Loss: {test_loss_two:.4f}")


**Two-Layer NN Results:**

- **Test Accuracy:** 0.3994 (~40%)

Adding a hidden layer improved accuracy by about 9.4 percentage points over the single-layer model. The non-linear ReLU activation allows the network to learn more complex feature combinations, enabling better separation of the 10 land-cover classes. Additional hidden layers generally improve accuracy by allowing the model to learn hierarchical representations — simple features in early layers combine into more complex patterns in later layers. However, too many layers can also worsen accuracy (as seen in Section 3.1.3) through overfitting or optimization difficulties if not carefully regularized.



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(history_two.history["accuracy"], label="Train Accuracy")
plt.plot(history_two.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Two-Layer Neural Network Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_two.history["loss"], label="Train Loss")
plt.plot(history_two.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Two-Layer Neural Network Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

from tensorflow.keras.utils import plot_model

plot_model(
    two_layer_model,
    show_shapes=True,
    show_layer_names=True,
    dpi=70
)

##### 3.1.3 Four-Layer Neural Network with Dropout

The architecture is deepened to four dense layers with dropout regularization:
**Input (4096) → Dense(256, ReLU) → Dropout(0.3) → Dense(128, ReLU) → Dropout(0.3) → Dense(64, ReLU) → Dropout(0.3) → Dense(10, softmax)**

This model has 1,090,634 trainable parameters — over 2× the two-layer model. Dropout randomly deactivates 30% of neurons during each training step to reduce co-adaptation and improve generalization.

In [ ]:
# define 4-layer model
from tensorflow import keras
from tensorflow.keras import layers

four_layer_model = keras.Sequential([
    layers.Input(shape=(4096,)),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(len(classes), activation="softmax")
])

four_layer_model.summary()

In [ ]:
four_layer_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# early_stop = EarlyStopping(
#     monitor="val_loss",
#     patience=4,
#     restore_best_weights=True
# )

history_four = four_layer_model.fit(
    X_train_gray_flat_nn,
    y_train_cat,
    validation_split=0.2,
    epochs=50,
    batch_size=128,
    verbose=1
)

In [ ]:
test_loss_four, test_acc_four = four_layer_model.evaluate(
    X_test_gray_flat_nn,
    y_test_cat,
    verbose=0
)

print(f"Four-Layer NN with Dropout Test Accuracy: {test_acc_four:.4f}")
print(f"Four-Layer NN with Dropout Test Loss: {test_loss_four:.4f}")


**Four-Layer NN with Dropout Results:**

- **Test Accuracy:** 0.3478 (~34.8%)

Counter-intuitively, the four-layer model performed *worse* than the two-layer model despite having more parameters. This reflects the difficulty of training deeper fully-connected networks on raw flattened pixel features: the input lacks spatial structure (it's a 1D vector of pixels), so adding more layers does not extract hierarchically richer features. Dropout further reduced effective capacity during training, preventing the model from memorizing training patterns but also slowing convergence. Dropout is most effective when the model has sufficient representational capacity to overfit; here the grayscale pixel features may not provide enough signal for deeper networks to leverage.



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(history_four.history["accuracy"], label="Train Accuracy")
plt.plot(history_four.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Four-Layer Neural Network with Dropout Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_four.history["loss"], label="Train Loss")
plt.plot(history_four.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Four-Layer Neural Network with Dropout Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

from tensorflow.keras.utils import plot_model

plot_model(
    four_layer_model,
    show_shapes=True,
    show_layer_names=True,
    dpi=70
)


##### 3.1.4 Model Comparison and Ensemble

The three fully connected models are compared on parameter count, test accuracy, and validation curves across epochs. An ensemble model is then built by **averaging the predicted class probability vectors** from all three models (soft voting), and its accuracy is evaluated on the test set.

In [ ]:
# compare model params
params_single = single_layer_model.count_params()
params_two = two_layer_model.count_params()
params_four = four_layer_model.count_params()

print(f"Single-layer params : {params_single:,}")
print(f"Two-layer params    : {params_two:,}")
print(f"Four-layer params   : {params_four:,}")

print("\nMargins:")
print(f"Two - Single  : {params_two - params_single:,}")
print(f"Four - Two    : {params_four - params_two:,}")
print(f"Four - Single : {params_four - params_single:,}")

In [ ]:
# compare accuracy
print(f"Single-Layer Test Accuracy        : {test_acc_single:.4f}")
print(f"Two-Layer Test Accuracy           : {test_acc_two:.4f}")
print(f"Four-Layer NN + Dropout Accuracy  : {test_acc_four:.4f}")

In [ ]:
# compare epoch
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

plt.plot(history_single.history["val_accuracy"], label="Single-Layer Val Acc")
plt.plot(history_two.history["val_accuracy"], label="Two-Layer Val Acc")
plt.plot(history_four.history["val_accuracy"], label="Four-Layer+Dropout Val Acc")

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy Across Epochs")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# compare log curves
plt.figure(figsize=(7, 5))

plt.plot(history_single.history["val_loss"], label="Single-Layer Val Loss")
plt.plot(history_two.history["val_loss"], label="Two-Layer Val Loss")
plt.plot(history_four.history["val_loss"], label="Four-Layer+Dropout Val Loss")

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss Across Epochs")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# build essemble
# predicted class probabilities on test set
pred_single = single_layer_model.predict(X_test_gray_flat_nn, verbose=0)
pred_two = two_layer_model.predict(X_test_gray_flat_nn, verbose=0)
pred_four = four_layer_model.predict(X_test_gray_flat_nn, verbose=0)

print(pred_single.shape, pred_two.shape, pred_four.shape)  # should all be (N, 10)


# simple average ensemble
pred_ensemble = (pred_single + pred_two + pred_four) / 3.0

# final predicted class
y_pred_ensemble = pred_ensemble.argmax(axis=1)

# if y_test_full is still integer labels, use directly
from sklearn.metrics import accuracy_score

ensemble_acc = accuracy_score(y_test_full, y_pred_ensemble)
print(f"Ensemble Test Accuracy: {ensemble_acc:.4f}")



print(f"Single-Layer Accuracy       : {test_acc_single:.4f}")
print(f"Two-Layer Accuracy          : {test_acc_two:.4f}")
print(f"Four-Layer + Dropout Acc    : {test_acc_four:.4f}")
print(f"Ensemble Accuracy           : {ensemble_acc:.4f}")


**Model Comparison Summary:**

| Model | Parameters | Test Accuracy |
|---|---|---|
| Single-Layer NN | 40,970 | 0.3055 |
| Two-Layer NN | 525,706 | **0.3994** |
| Four-Layer NN + Dropout | 1,090,634 | 0.3478 |
| Ensemble (avg of all 3) | — | 0.3916 |

**Which model had the most parameters?** The Four-Layer NN had 1,090,634 parameters — 564,928 more than the Two-Layer NN and 1,049,664 more than the Single-Layer NN.

**Which model was "best"?** The **Two-Layer NN** achieved the highest individual test accuracy (39.94%). Despite having fewer parameters than the four-layer model, it benefited from a well-matched capacity to the difficulty of the task.

**Impact of training epochs:** All three models showed slow, steady improvement in validation accuracy across epochs with no strong convergence before epoch 50. The single-layer model showed the least improvement per epoch (capacity-limited), while the two-layer model showed the most consistent gains.

**Ensemble:** The soft-voting ensemble (0.3916) slightly underperformed the best individual model (two-layer, 0.3994) but outperformed the weaker two. Ensembling works best when constituent models make *different* mistakes; since all three use the same flattened greyscale input, their errors are correlated, limiting the benefit.



#### 3.2 RGB Images

For this section, all 10 EuroSAT classes are used with **RGB images** (Sentinel-2 bands B04=Red, B03=Green, B02=Blue). Images are normalized per-image to [0, 1] and retain their 2D spatial structure (64×64×3) for input to convolutional models.

##### 3.2.1 CNN Model

A Convolutional Neural Network (CNN) is implemented with the following architecture:
**Conv2D(32, 3×3, ReLU) → MaxPooling2D(2×2) → Dropout(0.25) → Flatten → Dense(128, ReLU) → Dropout(0.5) → Dense(10, softmax)**

Unlike fully connected models, the CNN processes the image in its native spatial form, allowing convolutional filters to detect local spatial features (edges, textures, patterns) that are position-invariant.

In [ ]:
from tqdm.notebook import tqdm

def build_rgb_dataset(sample_list, desc="Preparing RGB dataset"):
    X_rgb, y_out = [], []
    for path, lbl in tqdm(sample_list, desc=desc):
        ms = load_ms_array(path) # (13, H, W)
        rgb = ms_to_rgb(ms)    # (H, W, 3) - still raw values
        # Normalize RGB to [0, 1]
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
        X_rgb.append(rgb)
        y_out.append(lbl)
    return np.stack(X_rgb), np.array(y_out, dtype=np.int32)

X_rgb_train_full, y_train_full_rgb = build_rgb_dataset(
    train_samples, desc="Preparing full TRAIN RGB dataset"
)
X_rgb_test_full, y_test_full_rgb = build_rgb_dataset(
    test_samples, desc="Preparing full TEST RGB dataset"
)

print("X_rgb_train_full shape:", X_rgb_train_full.shape)
print("X_rgb_test_full shape :", X_rgb_test_full.shape)
print("y_train_full_rgb shape     :", y_train_full_rgb.shape)
print("y_test_full_rgb shape      :", y_test_full_rgb.shape)


In [ ]:
#shuffle training set for class balance across train and validation sets
from sklearn.utils import shuffle

X_rgb_train_full, y_train_full_rgb = shuffle(
    X_rgb_train_full,
    y_train_full_rgb,
    random_state=42
)

In [ ]:
# One-hot encode the labels for the RGB dataset (y_train_full_rgb and y_test_full_rgb)
from tensorflow.keras.utils import to_categorical

num_classes = 10
y_train_cat_rgb = to_categorical(y_train_full_rgb, num_classes=num_classes)
y_test_cat_rgb = to_categorical(y_test_full_rgb, num_classes=num_classes)

print(y_train_cat_rgb.shape, y_test_cat_rgb.shape)


In [ ]:

import numpy as np

def check_class_balance(y, split_ratio=0.8):
    split_idx = int(len(y) * split_ratio)

    y_first = y[:split_idx]
    y_last  = y[split_idx:]

    def print_dist(name, y_subset):
        counts = np.bincount(y_subset)
        probs = counts / counts.sum()

        print(f"\n{name}:")
        for cls, p in enumerate(probs):
            print(f"Class {cls}: {p:.3f}")

    print_dist("First 80%", y_first)
    print_dist("Last 20%", y_last)

In [ ]:
#sanity check
check_class_balance(np.argmax(y_train_cat_rgb, axis=1))

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import plot_model

# Define the simplified CNN model with one of each specified layer type
cnn_model = keras.Sequential([
    layers.Input(shape=(64, 64, 3)), # Input shape for RGB images

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'), # One Conv2D layer
    layers.MaxPooling2D((2, 2)), # One MaxPooling2D layer
    layers.Dropout(0.25), # One Dropout layer after pooling

    layers.Flatten(), # One Flatten layer

    layers.Dense(128, activation='relu'), # One Dense hidden layer
    layers.Dropout(0.5), # Added dropout before the last dense layer
    layers.Dense(num_classes, activation='softmax') # Output Dense layer
])

cnn_model.summary()

# Visualize the simplified model architecture
plot_model(
    cnn_model,
    show_shapes=True,
    show_layer_names=True,
    dpi=70
)


In [ ]:
# Compile the CNN model
cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Early stopping to prevent overfitting
# early_stop_cnn = EarlyStopping(
#     monitor='val_loss',
#     patience=5,
#     restore_best_weights=True
# )

history_cnn = cnn_model.fit(
    X_rgb_train_full,
    y_train_cat_rgb,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    shuffle=True,
    verbose=1
)


In [ ]:
# Evaluate the CNN model on the test data
test_loss_cnn, test_acc_cnn = cnn_model.evaluate(
    X_rgb_test_full,
    y_test_cat_rgb,
    verbose=0
)

print(f"CNN Model Test Accuracy: {test_acc_cnn:.4f}")
print(f"CNN Model Test Loss: {test_loss_cnn:.4f}")


In [ ]:
import matplotlib.pyplot as plt

# Plot training history for CNN
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['accuracy'], label='Train Accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('CNN Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['loss'], label='Train Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.title('CNN Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import pickle

#save model
cnn_model.save("cnn_model_rgb.h5")

# Save history
with open("cnn_history_rgb.pkl", "wb") as f:
    pickle.dump(history_cnn.history, f)

In [ ]:
# Compare CNN with previous models
print(f"Single-Layer NN Test Accuracy       : {test_acc_single:.4f}")
print(f"Two-Layer NN Test Accuracy          : {test_acc_two:.4f}")
print(f"Four-Layer NN with Dropout Accuracy : {test_acc_four:.4f}")
print(f"CNN Model Test Accuracy             : {test_acc_cnn:.4f}")



**CNN Results and Comparison:**

| Model | Test Accuracy |
|---|---|
| Single-Layer NN (grayscale) | 0.3055 |
| Two-Layer NN (grayscale) | 0.3994 |
| Four-Layer NN + Dropout (grayscale) | 0.3478 |
| Ensemble (grayscale) | 0.3916 |
| **CNN (RGB)** | **0.7246** |

The CNN achieves 72.46% accuracy — a dramatic improvement (~32 points) over the best fully connected model. CNNs preserve and exploit **spatial structure**: convolutional filters slide across the image to detect local patterns (edges, textures, structures) regardless of where they appear. Fully connected models treat each pixel independently and lose all spatial relationships when the image is flattened.

**Spatial information handling:** CNNs use shared-weight convolutional filters to scan the entire image, learning position-invariant feature detectors. This is ideal for satellite imagery where the same texture pattern (e.g., a row of crop furrows, a road grid) may appear anywhere in the image. Fully connected models have no concept of spatial locality.

**Training speed:** Each CNN epoch took ~100–150 seconds vs. 1–9 seconds for the dense models. CNNs require more computation because convolution is applied at every spatial position, generating large intermediate feature maps (64×64×32 after the first Conv2D).



##### 3.2.2 Advanced Model

A deeper CNN with **Batch Normalization** is implemented to improve on the baseline CNN. The architecture adds a second convolutional block and batch normalization after each conv/dense layer:

**Conv2D(32)→BN→ReLU→MaxPool → Conv2D(64)→BN→ReLU→MaxPool→Dropout(0.25) → Flatten → Dense(128)→BN→ReLU→Dropout(0.5) → Dense(10, softmax)**

Batch normalization normalizes each layer's activations to have zero mean and unit variance during training, stabilizing and accelerating learning. Early stopping (patience=10) on validation loss is used to prevent overfitting across up to 100 epochs.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

advanced_cnn_model = keras.Sequential([
    layers.Input(shape=(64, 64, 3)),
    layers.Conv2D(32, (3, 3), activation=None, padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation=None, padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(128, activation=None),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax') # Adjusted for 10-class classification
])

advanced_cnn_model.summary()

In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(
    advanced_cnn_model,
    show_shapes=True,
    show_layer_names=True,
    dpi=70
)

In [ ]:
import tensorflow as tf

advanced_cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop_advanced_cnn = EarlyStopping(
    monitor='val_loss',
    patience=10, # Increased patience for potentially deeper model
    restore_best_weights=True
)

history_advanced_cnn = advanced_cnn_model.fit(
    X_rgb_train_full, # Using original full RGB data
    y_train_cat_rgb,
    validation_split=0.2,
    epochs=100, # Increased epochs as early stopping will manage overfitting
    batch_size=64,
    shuffle=True,
    callbacks=[early_stop_advanced_cnn],
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

# Evaluate the advanced CNN model on the test data
test_loss_advanced_cnn, test_acc_advanced_cnn = advanced_cnn_model.evaluate(
    X_rgb_test_full, # Using original full RGB data
    y_test_cat_rgb,
    verbose=0
)

print(f"Advanced CNN Model Test Accuracy: {test_acc_advanced_cnn:.4f}")
print(f"Advanced CNN Model Test Loss: {test_loss_advanced_cnn:.4f}")

# Plot training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_advanced_cnn.history['accuracy'], label='Train Accuracy')
plt.plot(history_advanced_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('Advanced CNN Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_advanced_cnn.history['loss'], label='Train Loss')
plt.plot(history_advanced_cnn.history['val_loss'], label='Validation Loss')
plt.title('Advanced CNN Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


**Advanced CNN Results:**

- **Test Accuracy: 0.8350 (83.5%)**

The advanced CNN improves by ~11 percentage points over the simple CNN, achieving 83.5% accuracy. The key improvements were:
1. **Additional convolutional block (Conv2D 64 filters):** Learns higher-level spatial features from the pooled output of the first block
2. **Batch normalization:** Stabilizes gradient flow through deeper layers, enabling faster and more reliable convergence
3. **Early stopping:** Prevented overfitting that would otherwise occur over 100 epochs

**Techniques chosen and rationale:** Batch normalization addresses internal covariate shift in deep networks. The second conv block provides two levels of spatial feature hierarchy (low-level edges/textures in block 1, higher-level structural patterns in block 2).

**Two classes with highest labeling error:** Based on the model's confusion patterns, **AnnualCrop vs PermanentCrop** and **Residential vs Industrial** have the highest misclassification rates in RGB. Both pairs are visually similar: annual and permanent crops share agricultural texture in RGB, while residential and industrial areas both contain buildings and impervious surfaces. Strategies to address this include using NIR/SWIR bands (which better discriminate vegetation type and building material) — which is exactly what Section 3.3 investigates.



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(history_single.history['val_accuracy'], label='Single-Layer NN')
plt.plot(history_two.history['val_accuracy'], label='Two-Layer NN')
plt.plot(history_four.history['val_accuracy'], label='Four-Layer NN')
plt.plot(history_cnn.history['val_accuracy'], label='Simple CNN')
plt.plot(history_advanced_cnn.history['val_accuracy'], label='Advanced CNN')

plt.title('Model Comparison: Validation Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(history_single.history['val_loss'], label='Single-Layer NN')
plt.plot(history_two.history['val_loss'], label='Two-Layer NN')
plt.plot(history_four.history['val_loss'], label='Four-Layer NN')
plt.plot(history_cnn.history['val_loss'], label='Simple CNN')
plt.plot(history_advanced_cnn.history['val_loss'], label='Advanced CNN')

plt.title('Model Comparison: Validation Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


**Overall Model Comparison:** The validation accuracy plots confirm a clear hierarchy: all greyscale FC models plateau below 40%, the simple CNN reaches ~72%, and the advanced CNN peaks at ~83%. The advanced CNN also converges faster in accuracy-per-epoch relative to the FC models.



#### 3.3 Multispectral Images

The **advanced CNN architecture** from Section 3.2.2 is adapted to accept 4-channel input. The selected bands are:

| Index | Band | Wavelength | Information |
|---|---|---|---|
| 3 | B04 (Red) | 665 nm | Vegetation, bare soil |
| 2 | B03 (Green) | 560 nm | Vegetation vigor |
| 1 | B02 (Blue) | 490 nm | Atmospheric, water |
| 7 | B08 (NIR) | 842 nm | Vegetation density, biomass |

The NIR band (B08) is the key addition beyond RGB. Vegetation reflects very strongly in the NIR while built-up surfaces and water reflect weakly, making this band highly discriminative for separating vegetated land cover classes from urban and water classes. The model input shape is adjusted from (64, 64, 3) to (64, 64, 4) — all other architectural details are identical to the advanced RGB CNN. Each band is independently normalized to [0, 1] per image.

In [ ]:
from tqdm.notebook import tqdm

def build_multispectral_dataset(sample_list, desc="Preparing Multispectral dataset"):
    X_ms, y_out = [], []
    # Bands: B04 (Red), B03 (Green), B02 (Blue), B08 (NIR)
    # Indices: 3, 2, 1, 7 (assuming 0-indexed multispectral array)
    SELECTED_BANDS = [3, 2, 1, 7]

    for path, lbl in tqdm(sample_list, desc=desc):
        ms = load_ms_array(path)  # (13, H, W)

        # Select desired bands and stack them (H, W, num_bands)
        selected_ms_bands = ms[SELECTED_BANDS, :, :]
        selected_ms_bands = np.transpose(selected_ms_bands, (1, 2, 0)) # (H, W, num_bands)

        # Normalize each band to [0, 1]
        normalized_ms = np.zeros_like(selected_ms_bands, dtype=np.float32)
        for i in range(selected_ms_bands.shape[-1]):
            band = selected_ms_bands[:, :, i]
            if band.max() - band.min() > 1e-6: # Avoid division by zero
                normalized_ms[:, :, i] = (band - band.min()) / (band.max() - band.min())
            else:
                normalized_ms[:, :, i] = 0.0 # Or some other reasonable default

        X_ms.append(normalized_ms)
        y_out.append(lbl)
    return np.stack(X_ms), np.array(y_out, dtype=np.int32)

print("Building multispectral datasets...")
X_ms_train_full, y_train_full_ms = build_multispectral_dataset(
    train_samples, desc="Preparing full TRAIN MS dataset"
)
X_ms_test_full, y_test_full_ms = build_multispectral_dataset(
    test_samples, desc="Preparing full TEST MS dataset"
)

print("\nX_ms_train_full shape:", X_ms_train_full.shape)
print("X_ms_test_full shape :", X_ms_test_full.shape)
print("y_train_full_ms shape     :", y_train_full_ms.shape)
print("y_test_full_ms shape      :", y_test_full_ms.shape)

In [ ]:
from sklearn.utils import shuffle

X_ms_train_full, y_train_full_ms = shuffle(
    X_ms_train_full,
    y_train_full_ms,
    random_state=42
)

print("Multispectral training data shuffled.")

In [ ]:
from tensorflow.keras.utils import to_categorical

num_classes = 10
y_train_cat_ms = to_categorical(y_train_full_ms, num_classes=num_classes)
y_test_cat_ms = to_categorical(y_test_full_ms, num_classes=num_classes)

print("Multispectral labels one-hot encoded.")
print("y_train_cat_ms shape:", y_train_cat_ms.shape)
print("y_test_cat_ms shape :", y_test_cat_ms.shape)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import plot_model

# Define a new Advanced CNN model for multispectral data
# Input shape now needs to reflect the number of selected bands (6 in this case)
advanced_cnn_ms_model = keras.Sequential([
    layers.Input(shape=(64, 64, X_ms_train_full.shape[-1])), # Dynamic input shape based on last dim of X_ms_train_full
    layers.Conv2D(32, (3, 3), activation=None, padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation=None, padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(128, activation=None),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

advanced_cnn_ms_model.summary()

plot_model(
    advanced_cnn_ms_model,
    show_shapes=True,
    show_layer_names=True,
    dpi=70
)

In [ ]:
import tensorflow as tf

advanced_cnn_ms_model.compile(
    optimizer='adam',
    loss='binary_crossentropy', # Binary cross-entropy for multi-label classification (one-hot encoded)
    metrics=['accuracy']
)

print("Advanced CNN for Multispectral data compiled.")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop_advanced_cnn_ms = EarlyStopping(
    monitor='val_loss',
    patience=10, # Increased patience for potentially deeper model
    restore_best_weights=True
)

history_advanced_cnn_ms = advanced_cnn_ms_model.fit(
    X_ms_train_full,
    y_train_cat_ms,
    validation_split=0.2,
    epochs=100, # Increased epochs as early stopping will manage overfitting
    batch_size=16, #reduce batch size to not overload RAM
    shuffle=True,
    callbacks=[early_stop_advanced_cnn_ms],
    verbose=1
)

print("Training of Advanced CNN for Multispectral data complete.")

In [ ]:
import matplotlib.pyplot as plt

# Evaluate the advanced CNN multispectral model on the test data
test_loss_advanced_cnn_ms, test_acc_advanced_cnn_ms = advanced_cnn_ms_model.evaluate(
    X_ms_test_full,
    y_test_cat_ms,
    verbose=0
)

print(f"Advanced CNN Multispectral Model Test Accuracy: {test_acc_advanced_cnn_ms:.4f}")
print(f"Advanced CNN Multispectral Model Test Loss: {test_loss_advanced_cnn_ms:.4f}")

# Plot training history for the multispectral model
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_advanced_cnn_ms.history['accuracy'], label='Train Accuracy')
plt.plot(history_advanced_cnn_ms.history['val_accuracy'], label='Validation Accuracy')
plt.title('Advanced CNN Multispectral Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_advanced_cnn_ms.history['loss'], label='Train Loss')
plt.plot(history_advanced_cnn_ms.history['val_loss'], label='Validation Loss')
plt.title('Advanced CNN Multispectral Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


**Multispectral (RGB + NIR) Model Results:**

- **Test Accuracy: 0.9094 (90.9%)**

Adding just one additional band — NIR — improved accuracy from 83.5% (RGB) to 90.9%, a gain of **+7.4 percentage points**. This is a significant improvement from a single additional channel, highlighting the value of NIR for land cover discrimination. NIR is particularly informative for distinguishing dense vegetation (Forest, HerbaceousVegetation) from spectrally similar classes in the visible spectrum (PermanentCrop, AnnualCrop). The model trained for 27 epochs before early stopping triggered.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 1. Get predictions (probabilities)
y_pred_probs = advanced_cnn_ms_model.predict(X_ms_test_full)

# 2. Convert to class indices
y_pred = np.argmax(y_pred_probs, axis=1)

# 3. Convert true labels from one-hot to class indices
y_true = np.argmax(y_test_cat_ms, axis=1)

# 4. Define label order and names (all 10 classes)
label_order = list(range(10))
label_names = [classes[i] for i in label_order]

# 5. Compute confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=label_order)

# 6. Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, cmap="Blues", colorbar=False)

plt.title("Confusion Matrix: Advanced CNN (Multispectral)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ---- Optional: Normalized version (very useful for analysis) ----
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=label_names)

fig, ax = plt.subplots(figsize=(8, 7))
disp_norm.plot(ax=ax, cmap="Blues", colorbar=True)

plt.title("Normalized Confusion Matrix: Advanced CNN (Multispectral)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Confusion Matrix Analysis:**

The confusion matrices reveal the residual error structure after adding NIR. Key observations:
- **PermanentCrop** has the lowest per-class accuracy and is the most confused class, frequently misclassified as AnnualCrop or HerbaceousVegetation — all three are vegetated surfaces with similar spectral appearance even in NIR
- **Forest** and **Residential** are well-separated with high accuracy (~97%), confirming that NIR cleanly separates dense canopy from urban surfaces
- **Highway** shows moderate confusion with Residential and Industrial — roads within urban areas are spectrally similar to building rooftops in all four bands
- The normalized confusion matrix makes it easy to see that off-diagonal mass is concentrated in the crop/vegetation cluster (AnnualCrop, HerbaceousVegetation, PermanentCrop) where NIR alone cannot fully resolve the within-vegetation distinctions



In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc
)
import matplotlib.pyplot as plt

# Convert predictions and true labels to class indices
y_true = np.argmax(y_test_cat_ms, axis=1)
y_pred_probs = advanced_cnn_ms_model.predict(X_ms_test_full)
y_pred = np.argmax(y_pred_probs, axis=1)

num_classes = y_test_cat_ms.shape[1]

# Store per-class metrics
per_class_accuracy = []
per_class_precision = []
per_class_recall = []
per_class_auc = []

for i in range(num_classes):
    # Mask for class i
    mask = (y_true == i)

    # Accuracy on samples of this class
    acc = accuracy_score(y_true[mask], y_pred[mask])

    # Precision & recall (one-vs-all)
    prec = precision_score(y_true == i, y_pred == i, average='binary', zero_division=0)
    rec = recall_score(y_true == i, y_pred == i, average='binary', zero_division=0)

    # AUC using probabilities
    try:
        auc_score = roc_auc_score(y_test_cat_ms[:, i], y_pred_probs[:, i])
    except ValueError:  # if class i missing in test set
        auc_score = np.nan

    per_class_accuracy.append(acc)
    per_class_precision.append(prec)
    per_class_recall.append(rec)
    per_class_auc.append(auc_score)

# Compute macro F1
macro_f1 = f1_score(y_true, y_pred, average='macro')

# Print per-class metrics
print(f"{'Class':<10} {'Acc':>6} {'Prec':>6} {'Recall':>6} {'AUC':>6}")
for i in range(num_classes):
    print(f"{classes[i]:<10} {per_class_accuracy[i]:.3f} {per_class_precision[i]:.3f} {per_class_recall[i]:.3f} {per_class_auc[i]:.3f}")

print(f"\nMacro F1 score (all classes): {macro_f1:.3f}")

# Optional: Plot ROC curves for all classes
plt.figure(figsize=(8, 6))
for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_test_cat_ms[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{classes[i]} (AUC={roc_auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--')  # diagonal
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves per Class")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

<font color='turquoise'>

**Per-Class Metrics Summary:**

| Class | Accuracy | Precision | Recall | AUC |
|---|---|---|---|---|
| AnnualCrop | 0.911 | 0.881 | 0.911 | 0.993 |
| Forest | 0.975 | 0.965 | 0.975 | 0.999 |
| HerbaceousVegetation | 0.833 | 0.884 | 0.833 | 0.990 |
| Highway | 0.881 | 0.858 | 0.881 | 0.987 |
| Industrial | 0.942 | 0.897 | 0.942 | 0.997 |
| Pasture | 0.919 | 0.904 | 0.919 | 0.997 |
| **PermanentCrop** | **0.766** | **0.835** | **0.766** | **0.982** |
| Residential | 0.973 | 0.923 | 0.973 | 0.999 |
| River | 0.928 | 0.940 | 0.928 | 0.995 |
| SeaLake | 0.949 | 0.984 | 0.949 | 0.999 |
| **Macro F1** | | | **0.907** | |

**Two classes with the highest labeling error:**

1. **PermanentCrop** (accuracy = 0.766, lowest of all classes): Permanent crops — orchards, vineyards — have a heterogeneous canopy structure that blends spectral properties of annual crops and herbaceous vegetation. In a 64×64 patch, the irregular spacing of tree/vine rows makes it hard to distinguish from dense annual crop fields, even with NIR. The AUC of 0.982 confirms the model can rank them well probabilistically, but the decision boundary is ambiguous.

2. **HerbaceousVegetation** (accuracy = 0.833): Meadows and grasslands are easily confused with AnnualCrop fields at certain growth stages, and with Pasture. All three classes consist of low, dense green vegetation and have very similar NIR reflectance profiles.

**Why?** Both confusion pairs involve vegetation classes that differ primarily in *land management* (how the vegetation is cultivated/maintained) rather than in *spectral reflectance*. The NIR band captures biomass/greenness but not crop management patterns. SWIR bands would help here — they are sensitive to vegetation water content and canopy structure, which differ between crop types.

**Strategies to address this:** (1) Add SWIR bands (B11, B12) to distinguish vegetation water status; (2) Add red-edge bands (B05, B06, B07) which are highly sensitive to chlorophyll and canopy structure differences between crop types; (3) Use larger spatial context (larger patch size or multi-scale inputs) to capture field boundary patterns; (4) Apply class-specific augmentation or class-weighted loss to compensate for the PermanentCrop confusion.

**ROC Curves:** All classes achieve AUC ≥ 0.982, confirming that the model's probability estimates are well-calibrated even for the harder classes. The lowest AUC is PermanentCrop (0.982) and HerbaceousVegetation (0.990) — consistent with the per-class accuracy analysis.

</font>

In [ ]:
import pickle
from tensorflow.keras.models import load_model
import numpy as np

# ---- 1. Load the saved RGB CNN model ----
cnn_model_rgb = load_model("cnn_model_rgb.h5")

# ---- 2. Load the saved training history ----
with open("cnn_history_rgb.pkl", "rb") as f:
    history_cnn_rgb = pickle.load(f)

# ---- 3. Evaluate on test data ----
test_loss_rgb, test_acc_rgb = cnn_model_rgb.evaluate(X_rgb_test_full, y_test_cat_rgb, verbose=0)

print(f"CNN RGB Test Accuracy: {test_acc_rgb:.4f}")
print(f"CNN RGB Test Loss: {test_loss_rgb:.4f}")

# ---- 4. Compare with multispectral CNN ----
test_loss_ms, test_acc_ms = advanced_cnn_ms_model.evaluate(X_ms_test_full, y_test_cat_ms, verbose=0)
print(f"Advanced CNN Multispectral Test Accuracy: {test_acc_ms:.4f}")
print(f"Advanced CNN Multispectral Test Loss: {test_loss_ms:.4f}")

# ---- 5. Quick comparison ----
accuracy_diff = test_acc_ms - test_acc_rgb
print(f"Accuracy improvement using multispectral vs RGB: {accuracy_diff:.4f}")


**Final Comparison: All Deep Learning Models**

| Model | Input | Test Accuracy |
|---|---|---|
| Single-Layer NN | Greyscale (flattened) | 0.3055 |
| Two-Layer NN | Greyscale (flattened) | 0.3994 |
| Four-Layer NN + Dropout | Greyscale (flattened) | 0.3478 |
| Ensemble (FC models 1–3) | Greyscale (flattened) | 0.3916 |
| Simple CNN | RGB (64×64×3) | 0.7246 |
| Advanced CNN | RGB (64×64×3) | 0.8350 |
| **Advanced CNN (best model)** | **RGB + NIR (64×64×4)** | **0.9094** |

**Conclusion:** The **Advanced CNN trained on RGB + NIR** is the best-performing model overall, achieving 90.9% test accuracy across all 10 land cover classes. The progression across the table tells a clear story about what drives performance gains:

1. **Architecture matters more than depth (for FC models):** Going from 1 to 2 layers gave a +9.4 point gain; going deeper to 4 layers with dropout actually hurt (-5.2 points), because raw flattened pixels lack the spatial structure that deeper networks need to build hierarchical representations from.

2. **Spatial structure is the biggest single lever:** Switching from a flat pixel vector to a 2D spatial input for the CNN — with otherwise comparable complexity — produced the largest single jump in the entire experiment (+32.5 points over the best FC model). Convolutional filters can detect local textures and edges regardless of where they appear in the image; fully connected models cannot.

3. **Spectral information compounds spatial gains:** Adding just one extra band (NIR, B08) on top of the advanced RGB CNN produced another +7.4 point improvement. NIR reflectance is driven by leaf cell structure and is fundamentally different information from anything visible-spectrum bands can provide, giving the model the ability to separate vegetation classes from built-up and water surfaces far more cleanly.

The remaining ~9% error is concentrated in the vegetation sub-classes (PermanentCrop, HerbaceousVegetation, AnnualCrop), which share similar NIR responses and differ mainly in land management patterns that require additional spectral bands (SWIR, red-edge) or larger spatial context to resolve.



### 4. Reflection Questions


**What are your takeaways from tuning the parameters of the different models?**

The clearest takeaway is that *what you feed the model matters far more than how you tune it*. Moving from greyscale flattened pixels to spatial RGB images — with no tuning at all — produced a 32-point accuracy jump that no amount of layer-count or dropout tuning in the FC section could approach. Within a given input representation, the most impactful tuning decisions were architectural: adding a second convolutional block and batch normalization produced a  ~11-point gain over the simple CNN. Hyperparameter choices like learning rate (Adam default) and batch size were less critical, though reducing batch size to 16 for the multispectral model was necessary to avoid memory issues and had an unintended side effect of noisier but ultimately comparable convergence.

**What are your observations about increasing the number of training epochs? Did you run into any challenges or limitations when doing this?**

For the fully connected models, validation accuracy improved slowly and steadily through all 50 epochs with no clear convergence plateau — suggesting these models were capacity-limited rather than converging, and more epochs would likely continue to help marginally. For the CNN models, early stopping proved essential: the advanced RGB CNN was allowed up to 100 epochs and stopped at 22 with best weights restored, while the multispectral model stopped at 27. Without early stopping, validation loss began creeping up even as training loss fell, indicating overfitting. The main challenge was wall-clock time: each epoch of the multispectral model took ~270–320 seconds on Colab (due to the small batch size of 16), making experimentation slow.

**What was the impact of using dropout?**

Dropout had opposite effects depending on the model type. In the four-layer FC model, dropout (0.3 rate on all hidden layers) *hurt* performance relative to the two-layer model without it, likely because the raw pixel features do not provide enough signal for a deeper FC network — dropout further starved the model of the weak signal available. In the CNN models, dropout was beneficial: the convolutional layers first extract spatially meaningful features, so there is something substantive to regularize, and dropout prevented the dense head from memorizing training examples. The lesson is that dropout works best when the upstream representations are rich enough that randomly dropping 30–50% of them still leaves useful information.

**How did the ensemble models compare to the other models?**

The soft-voting ensemble of the three greyscale FC models (accuracy 0.3916) outperformed the single-layer and four-layer models but fell just short of the two-layer model (0.3994). This is a common outcome when ensembling models that share the same input features: their errors are correlated, so averaging their predictions partially cancels out errors but cannot correct for the systematic confusions they all share. Ensembling would be more powerful if the constituent models used different inputs (e.g., greyscale + RGB + multispectral) or different architectures with genuinely diverse inductive biases.

**What kinds of challenges or limitations did you encounter when preparing and training the models for this assignment, and how might you address them in the future?**

Several practical challenges arose. First, **memory**: the full 43,200-sample dataset was too large to flatten and load as a dense numpy array for all 13 bands simultaneously; we addressed this by subsetting to three classes for traditional ML and using batch loading for deep learning. Second, **training time**: the multispectral CNN required ~5 hours of total training time on Colab's free GPU, limiting how many architectures we could experiment with. Third, **loss function mismatch**: binary cross-entropy was used for the multi-class CNN models (a technical error), though the models converged effectively despite this. In future work, categorical cross-entropy should be used for multi-class problems. Fourth, **data leakage awareness**: augmenting before splitting is a common mistake; building the split-first pipeline required careful sequencing of the preprocessing steps. Fifth, **class imbalance in the train/validation split**: Keras's `validation_split=0.2` takes the *last* 20% of the training array as the validation set, which means class balance in the train and validation portions depends entirely on how the data is ordered. We discovered that the default ordering produced uneven class distributions across the two portions — for example, the first 80% and last 20% had noticeably different per-class proportions. We addressed this by applying `sklearn.utils.shuffle` to the training set before fitting, then verified with a custom `check_class_balance` function that class proportions were approximately consistent between the first 80% and last 20% of the shuffled array. This is a subtle but important issue: a validation set that over- or under-represents certain classes will produce misleading validation accuracy curves during training, potentially causing early stopping to trigger at the wrong moment.

**How might you apply what you've learned about model tuning, dropout, and data processing to a different deep learning problem?**

The core lessons generalize well. For any spatially structured input (medical images, aerial photos, street-level imagery), CNNs should be the default starting point over FC networks, and input representation — which channels or modalities to include — should be treated as a primary design decision rather than an afterthought. The augmentation-after-splitting discipline applies to any dataset where transformed samples might bleed across train/test boundaries. Batch normalization + early stopping is a robust combination for preventing both slow convergence and overfitting in moderate-depth CNNs. The validation split class balance issue is a general lesson too: whenever using a sequential split for validation (as opposed to `StratifiedShuffleSplit`), always shuffle first and verify class distributions in both portions — or better yet, use Keras's `validation_data` argument with a pre-constructed stratified split to guarantee balance by construction. Finally, the result that a single well-chosen spectral band (NIR) produced a larger accuracy gain than any architectural change suggests that for domain-specific problems, investing time in understanding *what information the input encodes* often pays off more than architecture search alone.



<font color='turquoise'>

**Final RGB vs Multispectral Comparison:**

| Model | Input | Test Accuracy |
|---|---|---|
| Simple CNN | RGB (3 bands) | 0.7246 |
| Advanced CNN | RGB (3 bands) | 0.8350 |
| **Advanced CNN** | **RGB + NIR (4 bands)** | **0.9094** |
| Accuracy improvement (MS vs simple RGB CNN) | | +0.1848 |

Adding a single NIR band to the advanced CNN architecture produced a **+7.4 point improvement** over RGB alone (83.5% → 90.9%). This demonstrates that even one extra spectral channel, carefully chosen, can have a substantial impact on classification performance for satellite land cover mapping.

**Role of additional spectral information:** Different land cover types have characteristic *spectral signatures* across wavelengths. RGB data only covers 400–700 nm, capturing colour but missing the most diagnostically powerful spectral range for vegetation. NIR reflectance (around 842 nm) is driven by leaf cell structure and is high for healthy vegetation and low for water, bare soil, and built-up surfaces — creating natural separability that no RGB combination can replicate. This is why the normalized difference vegetation index (NDVI = (NIR − Red) / (NIR + Red)) has been a cornerstone of remote sensing for decades.

</font>

### 4. Reflection Questions

<font color='turquoise'>

**Parameter tuning takeaways:** The most impactful single decision across all models was the *choice of input representation*. Moving from flattened grayscale pixels to spatial RGB (CNN) produced a ~32-point accuracy gain; adding a single NIR band produced another ~7 points. Within deep learning models, batch normalization and a second convolutional block produced a consistent ~11-point gain over the simple CNN. For SVMs and Random Forests, default parameters performed well — Random Forest accuracy was stable across 50–200 estimators.

**Effect of training epochs:** All FC models showed continuous if slow improvement through epoch 50, suggesting more epochs could help but with diminishing returns. The CNN models converged faster and benefited from early stopping. The multispectral model trained for 27 epochs before early stopping triggered.

**Impact of dropout:** In the four-layer FC model, dropout reduced accuracy below the two-layer model — the raw pixel features don't provide enough signal for deeper FC networks, and dropout further reduced effective capacity. In the CNN models, dropout was more effective because the convolutional layers first extract spatially meaningful features, giving the regularizer something worthwhile to work with.

**Ensemble performance:** The soft-voting ensemble of the three greyscale FC models did not consistently outperform the best individual model. Ensembling is most effective when constituent models make diverse errors; since all three shared the same input features, their errors were correlated.

**Challenges encountered:** (1) Memory constraints with the full 43,200-sample dataset required a reduced batch size (16) for the multispectral model, which significantly increased training time per epoch. (2) The CNN models required substantial wall-clock time on Colab (~270–330s/epoch for the multispectral model). (3) Binary cross-entropy was used for the CNN models rather than categorical cross-entropy — a mismatch for multi-class classification that the models learned through despite.

**Future applications:** The workflow demonstrated here — stratified splits, augmentation-after-splitting, batch normalization, early stopping, adding informative spectral bands — applies directly to any satellite image classification problem (e.g., flood mapping, urban growth monitoring, deforestation detection). The lesson that spatial structure matters (CNN >> FC) holds for any grid-structured input. Adding red-edge and SWIR bands would likely push accuracy further, particularly for the vegetation sub-classes where the 4-band model still struggles.

</font>